### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="cooking_time",
    dataset_year="2024",
    domain_str="industry & manufacturing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/pcovkrd84mejm/cooking-time",
    download_description="""
We get the TabRed data from Kaggle.

kaggle datasets download -d pcovkrd84mejm/cooking-time -f cooking_time.parquet && unzip cooking_time.parquet.zip && rm cooking_time.parquet.zip
mkdir -p local-data-warehouse/cooking_time && mv cooking_time.parquet local-data-warehouse/cooking_time/
""",
    # References
    academic_reference_bibtex="""@inproceedings{rubachev2025tabred,
  title={TabReD: Analyzing Pitfalls and Filling the Gaps in Tabular Deep Learning Benchmarks},
  author={Rubachev, Ivan and Kartashev, Nikolay and Gorishniy, Yury and Babenko, Artem},
  booktitle={The Thirteenth International Conference on Learning Representations},
  year={2025},
}
""",
    academic_reference_bibtex_key="rubachev2025tabred",
    license="CC-BY-NC-SA-4.0",
    data_tags=["Non-IID", "Temporal", "Anonymized"],
    curation_comments="""
We start with data from TabRed, which already comes preprocessed.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="cooking_time_minutes",
    problem_type="regression",
    objective_metric_name="rmse",
    time_on="timestamp",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet(dataset_mold.path / "cooking_time.parquet")
print("Loaded data shape:", df.shape)

df["timestamp"] = pd.to_datetime(df["timestamp"])

# Following TabRed
df = df[df["cooking_time_minutes"] >= 1.0]

# We take all bin + cat as Category
cat_cols = [c for c in df.columns if c.startswith("cat") or c.startswith("bin")]
df[cat_cols] = df[cat_cols].astype("category")

df["cooking_time_minutes"] = np.log(df["cooking_time_minutes"])

df = df.sort_values(by="timestamp").reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata

date_col = task_mold.time_on
target_col = task_mold.target_column_name

df = df.sort_values(by=date_col).reset_index(drop=True)

test_time_min = df[date_col].max().normalize() - pd.DateOffset(weeks=1)

# Define indices
test_idx = df.index[df[date_col] >= test_time_min].to_numpy().tolist()
train_idx = df.index[
    df[date_col] < test_time_min
].to_numpy().tolist()

print("Train size:", len(train_idx), "| Test size:", len(test_idx))
print("Train target mean:", df.loc[train_idx, target_col].mean())
print("Test target mean:", df.loc[test_idx, target_col].mean())

splits = {0: {0: (train_idx, test_idx)}}

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We use the last week as test data and all prior data as train data.",
    splits=splits,
    time_horizon=7,
    time_horizon_unit="days",
)

## Export

In [ ]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)